# Kimi Audio Instruct — Zero-Shot Alzheimer's Detection

Using `moonshotai/Kimi-Audio-7B-Instruct` to classify speech samples as **Dementia** or **Control** via prompting (no training).

**Test conditions**: 2 raw + 10 denoised = 12 total (Pitt & Lu datasets × 5 denoising methods + raw).

In [ ]:
import subprocess
import os

result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
output = result.stdout
for line in output.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

In [ ]:
!pip install git+https://github.com/MoonshotAI/Kimi-Audio.git

In [ ]:
import json
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from kimia_infer.api.kimia import KimiAudio

CACHE_DIR    = "/root/autodl-tmp/LLM_Model"
PROJECT_ROOT = Path("../ad_detection")
MODEL_ID     = "moonshotai/Kimi-Audio-7B-Instruct"

In [ ]:
model = KimiAudio(model_path=MODEL_ID, load_detokenizer=True, cache_dir=CACHE_DIR)
print("Kimi Audio model loaded.")

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [ ]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="kimi_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [ ]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": [
                {"type": "audio", "audio_url": str(wav_path)},
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]
    _, text = model.generate(messages, audio_generate=False)
    return text

In [ ]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    for _, row in tqdm(df.iterrows(), total=len(df), desc=name):
        label_dir = label_map[row["ad"]]
        matches = list(Path(audio_dir).glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            pred = raw.strip() if raw.strip() in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/len(ctrl):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /len(dem) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/raw/Pitt"
evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Demucs"
evaluate_dataset(csv, audio_dir, "Pitt-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Demucs"
evaluate_dataset(csv, audio_dir, "Lu-Demucs")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Denoiser"
evaluate_dataset(csv, audio_dir, "Pitt-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Denoiser"
evaluate_dataset(csv, audio_dir, "Lu-Denoiser")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Pitt-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-FRCRN_SE"
evaluate_dataset(csv, audio_dir, "Lu-FRCRN_SE")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-MossFormer"
evaluate_dataset(csv, audio_dir, "Pitt-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-MossFormer"
evaluate_dataset(csv, audio_dir, "Lu-MossFormer")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Pitt-Resemble"
evaluate_dataset(csv, audio_dir, "Pitt-Resemble")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
audio_dir = PROJECT_ROOT / "data/denoised/Lu-Resemble"
evaluate_dataset(csv, audio_dir, "Lu-Resemble")